# 05 Validation

Validate the MVP `foods_master` dataset and export a quality report.

Outputs:
- `data/processed/validation_report.parquet`
- `data/processed/validation_report.json`

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'data' / 'processed'
FOODS_MASTER_PATH = OUTPUT_DIR / 'foods_master.parquet'
REPORT_PARQUET_PATH = OUTPUT_DIR / 'validation_report.parquet'
REPORT_JSON_PATH = OUTPUT_DIR / 'validation_report.json'

EXPECTED_COLUMNS = [
    'food_id', 'food_name_en', 'food_name_ar', 'food_group', 'meal_types',
    'serving_name', 'serving_weight_g', 'nutrition_basis', 'calories',
    'protein', 'carbs', 'fat', 'fiber', 'diet_tags', 'allergens',
    'is_composite_dish', 'source'
]
NUMERIC_COLUMNS = ['serving_weight_g', 'calories', 'protein', 'carbs', 'fat', 'fiber']
NUTRIENT_COLUMNS = ['calories', 'protein', 'carbs', 'fat', 'fiber']
TARGET_GROUPS = {
    'Grains', 'Vegetables', 'Fruits', 'Protein', 'Dairy', 'Legumes',
    'Healthy Fats', 'Composite Dish', 'Beverages', 'Snacks'
}

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}. Run notebook 04 first.')

def normalized_name(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def add_metric(records: list, section: str, metric: str, value: object, status: str = 'info') -> None:
    records.append({'section': section, 'metric': metric, 'value': value, 'status': status})

require_file(FOODS_MASTER_PATH)
df = pd.read_parquet(FOODS_MASTER_PATH, engine='pyarrow')
records = []

missing_columns = [col for col in EXPECTED_COLUMNS if col not in df.columns]
extra_columns = [col for col in df.columns if col not in EXPECTED_COLUMNS]
add_metric(records, 'schema', 'row_count', int(len(df)))
add_metric(records, 'schema', 'column_count', int(len(df.columns)))
add_metric(records, 'schema', 'missing_required_columns', '|'.join(missing_columns), 'fail' if missing_columns else 'pass')
add_metric(records, 'schema', 'extra_columns', '|'.join(extra_columns), 'fail' if extra_columns else 'pass')

for col in EXPECTED_COLUMNS:
    if col in df.columns:
        missing_count = int(df[col].isna().sum() + (df[col].astype(str).str.strip().eq('').sum() if df[col].dtype == object else 0))
        add_metric(records, 'missing_values', col, missing_count, 'warn' if missing_count else 'pass')

if 'food_name_en' in df.columns:
    name_keys = df['food_name_en'].map(normalized_name)
    duplicate_foods = int(name_keys.duplicated().sum())
    add_metric(records, 'duplicates', 'duplicated_food_name_en', duplicate_foods, 'warn' if duplicate_foods else 'pass')
if 'food_id' in df.columns:
    duplicate_ids = int(df['food_id'].duplicated().sum())
    add_metric(records, 'duplicates', 'duplicated_food_id', duplicate_ids, 'fail' if duplicate_ids else 'pass')

for col in NUMERIC_COLUMNS:
    if col in df.columns:
        values = pd.to_numeric(df[col], errors='coerce')
        add_metric(records, 'column_statistics', f'{col}_min', float(values.min()))
        add_metric(records, 'column_statistics', f'{col}_median', float(values.median()))
        add_metric(records, 'column_statistics', f'{col}_max', float(values.max()))
        add_metric(records, 'column_statistics', f'{col}_mean', float(values.mean()))

invalid_serving = int((pd.to_numeric(df.get('serving_weight_g', pd.Series(dtype=float)), errors='coerce') <= 0).sum())
add_metric(records, 'serving_sizes', 'invalid_serving_weight_g', invalid_serving, 'fail' if invalid_serving else 'pass')

if set(NUTRIENT_COLUMNS).issubset(df.columns):
    nutrients = df[NUTRIENT_COLUMNS].apply(pd.to_numeric, errors='coerce')
    invalid_macros = int(((nutrients[['protein', 'carbs', 'fat', 'fiber']] < 0).any(axis=1)).sum())
    macro_grams_over_100 = int((nutrients[['protein', 'carbs', 'fat', 'fiber']].sum(axis=1) > 120).sum())
    calories_negative = int((nutrients['calories'] < 0).sum())
    calories_too_high = int((nutrients['calories'] > 950).sum())
    protein_too_high = int((nutrients['protein'] > 100).sum())
    carbs_too_high = int((nutrients['carbs'] > 100).sum())
    fat_too_high = int((nutrients['fat'] > 100).sum())
    fiber_too_high = int((nutrients['fiber'] > 80).sum())
    add_metric(records, 'invalid_macros', 'negative_macro_rows', invalid_macros, 'fail' if invalid_macros else 'pass')
    add_metric(records, 'invalid_macros', 'macro_sum_over_120g_rows', macro_grams_over_100, 'warn' if macro_grams_over_100 else 'pass')
    add_metric(records, 'nutrition_outliers', 'negative_calories_rows', calories_negative, 'fail' if calories_negative else 'pass')
    add_metric(records, 'nutrition_outliers', 'calories_over_950_rows', calories_too_high, 'warn' if calories_too_high else 'pass')
    add_metric(records, 'nutrition_outliers', 'protein_over_100g_rows', protein_too_high, 'warn' if protein_too_high else 'pass')
    add_metric(records, 'nutrition_outliers', 'carbs_over_100g_rows', carbs_too_high, 'warn' if carbs_too_high else 'pass')
    add_metric(records, 'nutrition_outliers', 'fat_over_100g_rows', fat_too_high, 'warn' if fat_too_high else 'pass')
    add_metric(records, 'nutrition_outliers', 'fiber_over_80g_rows', fiber_too_high, 'warn' if fiber_too_high else 'pass')

if 'food_group' in df.columns:
    invalid_groups = int(~df['food_group'].isin(TARGET_GROUPS).sum()) if False else int((~df['food_group'].isin(TARGET_GROUPS)).sum())
    add_metric(records, 'categories', 'invalid_food_group_rows', invalid_groups, 'fail' if invalid_groups else 'pass')
    for group, count in df['food_group'].value_counts(dropna=False).sort_index().items():
        add_metric(records, 'categories', f'food_group_{group}', int(count))

if 'source' in df.columns:
    for source, count in df['source'].value_counts(dropna=False).sort_index().items():
        add_metric(records, 'sources', f'source_{source}', int(count))

report = pd.DataFrame(records)
report['value'] = report['value'].astype(str)
report.to_parquet(REPORT_PARQUET_PATH, index=False, engine='pyarrow')
with REPORT_JSON_PATH.open('w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'Validation metrics exported: {len(report)}')
print(f'Parquet report: {REPORT_PARQUET_PATH}')
print(f'JSON report: {REPORT_JSON_PATH}')
print(report)
